In [3]:
# Cell 1 — Setup + data load (synthetic rigged-edge fixture)
import pandas as pd
from pathlib import Path
from backtest.data import DataPanel

DATA_DIR = Path(r"C:\Users\User\backtest_engine\backtest_engine2\data")
PX_PATH  = DATA_DIR / "synth_edge_prices.parquet"
VOL_PATH = DATA_DIR / "synth_edge_volume.parquet"

prices = pd.read_parquet(PX_PATH)
volume = pd.read_parquet(VOL_PATH)

panel = DataPanel(prices, volume=volume, check_outliers=True)

print(f"Bars:   {len(panel.dates)}")
print(f"Assets: {len(panel.assets_all)}")
print(f"Range:  {panel.dates[0].date()} -> {panel.dates[-1].date()}")
print(f"NaN fraction in prices: {prices.isna().mean().mean():.3%}")
prices.tail(3).iloc[:, :6]


Bars:   1500
Assets: 20
Range:  2015-01-01 -> 2020-09-30
NaN fraction in prices: 0.000%
                   A00         A01         A02         A03         A04  \
Date                                                                     
2020-09-28  319.598188  543.363221  407.206344  329.154903  396.569318   
2020-09-29  318.766358  542.134950  406.708076  329.813837  398.310440   
2020-09-30  321.447285  541.325032  407.022910  328.214428  397.285586   

                   A05  
Date                    
2020-09-28  258.289863  
2020-09-29  258.316114  
2020-09-30  257.997764  


In [ ]:
# Cell 2 — Strategy: weekly dollar-neutral cross-sectional momentum L/S
import pandas as pd
from backtest.strategy import Strategy


class XSMomentum(Strategy):
    """Weekly cross-sectional momentum, dollar-neutral. Long the top-k trailing
    returns, short the bottom-k, at +/- 0.5/k per name (gross ~1.0, net 0).
    apply_risk is identity so the run isolates raw engine PnL + cost logic."""

    rebalance_frequency = "weekly"

    def __init__(self, lookback: int = 60, k: int = 5):
        self.lookback = lookback
        self.k = k

    def __repr__(self):
        return f"XSMomentum(lookback={self.lookback}, k={self.k})"

    def generate_weights(self, data, t):
        px = data.prices
        if len(px) < self.lookback + 1:
            return pd.Series(0.0, index=px.columns)
        trail = px.iloc[-1] / px.iloc[-self.lookback - 1] - 1.0
        ranked = trail.rank()
        w = pd.Series(0.0, index=px.columns)
        w.loc[ranked.nlargest(self.k).index] = +0.5 / self.k
        w.loc[ranked.nsmallest(self.k).index] = -0.5 / self.k
        return w

    def apply_risk(self, proposed, state, data):
        return proposed  # identity: no vol-targeting, no caps


strat = XSMomentum(lookback=60, k=5)
print(strat)
rd = strat.rebalance_dates(panel.dates)
print(f"Rebalances: {len(rd)}  |  per year: {len(rd) / (len(panel.dates) / 252):.1f}")
print(f"First 3 rebalance dates: {[str(d.date()) for d in rd[:3]]}")
print(f"Per-name weight: +/- {0.5 / strat.k:.2f}  |  gross {2 * strat.k * (0.5 / strat.k):.1f}, net 0")


XSMomentum(lookback=60, k=5)
Rebalances: 301  |  per year: 50.6
First 3 rebalance dates: ['2015-01-01', '2015-01-05', '2015-01-12']
Per-name weight: +/- 0.10  |  gross 1.0, net 0


In [ ]:
# Cell 3 — Costs + liquidity (mandatory before any engine run)
from backtest.costs import (
    Commission, Spread, MarketImpact, ShortBorrow,
    CompositeCostModel, LiquidityCap,
)

costs = CompositeCostModel([
    Commission(bps=2),                                  # 2 bps commission
    Spread(half_bps=2),                                 # 2 bps half-spread
    MarketImpact(k=10, kind="sqrt", adv_lookback=20),   # sqrt impact, 20-bar ADV
    ShortBorrow(annual_bps=300, trading_days=252),      # 3%/yr borrow on shorts (daily bars)
])

liq = LiquidityCap(cap_pct=0.10, adv_lookback=20)       # 10% of ADV (won't bind on this fixture)

print("Cost components:", [type(m).__name__ for m in costs.models])
print(f"LiquidityCap: cap_pct={liq.cap_pct}, adv_lookback={liq.adv_lookback}")
print(f"Panel has volume: {panel.has_field('volume')}")  # must be True for MarketImpact + LiquidityCap


Cost components: ['Commission', 'Spread', 'MarketImpact', 'ShortBorrow']
LiquidityCap: cap_pct=0.1, adv_lookback=20
Panel has volume: True


In [ ]:
# Cell 4 — Single-path Engine, GROSS run (costs=None, opt-in calibration per spec)
import numpy as np
from backtest.engine import Engine

WARMUP_BARS = 65  # spec warmup_bars; strategy guard needs lookback+1 = 61 bars

engine_gross = Engine(
    panel, strat,
    costs=None, liquidity=None,          # gross: isolate raw PnL logic (spec-requested opt-in)
    initial_capital=1_000_000,
)
result_gross = engine_gross.run(start=panel.dates[WARMUP_BARS])

r = result_gross.returns.dropna()
sharpe_rough = r.mean() / r.std() * np.sqrt(252)

print(f"Bars:          {len(result_gross.equity_curve)}")
print(f"Rebalances:    {result_gross.metadata['n_rebalances']}")
print(f"Date range:    {result_gross.metadata['start'].date()} -> {result_gross.metadata['end'].date()}")
print(f"Initial:       ${result_gross.metadata['initial_capital']:>14,.0f}")
print(f"Final equity:  ${result_gross.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return:  {(result_gross.equity_curve.iloc[-1] / result_gross.metadata['initial_capital'] - 1):+.2%}")
print(f"Total costs:   ${result_gross.costs.sum():>14,.0f}")
print(f"Max position weight:  {result_gross.weights.abs().max().max():.2%}")
print(f"Max gross exposure:   {result_gross.weights.abs().sum(axis=1).max():.2%}")
print(f"Mean net exposure:    {result_gross.weights.sum(axis=1).mean():+.4f}")
print(f"Rough Ann. Sharpe:    {sharpe_rough:+.2f}   (authoritative MetricsReport in Stage 7)")


Bars:          1435
Rebalances:    288
Date range:    2015-04-02 -> 2020-09-30
Initial:       $     1,000,000
Final equity:  $     3,707,451
Total return:  +270.75%
Total costs:   $             0
Max position weight:  10.37%
Max gross exposure:   100.91%
Mean net exposure:    +0.0018
Rough Ann. Sharpe:    +9.16   (authoritative MetricsReport in Stage 7)


In [ ]:
# Cell 5 — Single-path Engine, NET run (costs + liquidity on; mandatory)
import numpy as np
from backtest.engine import Engine

engine_net = Engine(
    panel, strat,
    costs=costs, liquidity=liq,          # from Cell 3 — mandatory
    initial_capital=1_000_000,
)
result_net = engine_net.run(start=panel.dates[WARMUP_BARS])

r = result_net.returns.dropna()
sharpe_rough = r.mean() / r.std() * np.sqrt(252)

init = result_net.metadata["initial_capital"]
print(f"Final equity:  ${result_net.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return:  {(result_net.equity_curve.iloc[-1] / init - 1):+.2%}   (gross was +270.75%)")
print(f"Total costs:   ${result_net.costs.sum():>14,.0f}   ({result_net.costs.sum() / init:.2%} of initial capital)")
print(f"Rough Ann. Sharpe:    {sharpe_rough:+.2f}   (gross was +9.16)")
print(f"Unfilled (liquidity): ${result_net.unfilled.abs().sum().sum():>14,.0f}   (expect 0 -> cap never binds)")
print(f"Max position weight:  {result_net.weights.abs().max().max():.2%}")
print(f"Max gross exposure:   {result_net.weights.abs().sum(axis=1).max():.2%}")
print(f"Mean net exposure:    {result_net.weights.sum(axis=1).mean():+.4f}")


Final equity:  $     3,288,284
Total return:  +228.83%   (gross was +270.75%)
Total costs:   $       239,977   (24.00% of initial capital)
Rough Ann. Sharpe:    +8.29   (gross was +9.16)
Unfilled (liquidity): $             0   (expect 0 -> cap never binds)
Max position weight:  10.38%
Max gross exposure:   100.94%
Mean net exposure:    +0.0018


In [ ]:
# Cell 6 — Metrics report (gross vs net), authoritative MetricsReport
from backtest.metrics import compute_metrics

rep_gross = compute_metrics(
    result_gross.returns, result_gross.equity_curve,
    costs=result_gross.costs, trades=result_gross.trades,
    ann_factor=252, ci_level=0.95,      # returns are daily even though rebalance is weekly
)
rep_net = compute_metrics(
    result_net.returns, result_net.equity_curve,
    costs=result_net.costs, trades=result_net.trades,
    ann_factor=252, ci_level=0.95,
)

print("=" * 30, "GROSS", "=" * 30)
print(rep_gross)
print("\n" + "=" * 30, "NET", "=" * 32)
print(rep_net)

print("\n" + "=" * 24, "GROSS vs NET (headline)", "=" * 24)
hdr = f"{'metric':<22}{'gross':>14}{'net':>14}"
print(hdr); print("-" * len(hdr))
def row(name, g, n, fmt="{:>14.3f}"):
    print(f"{name:<22}" + fmt.format(g) + fmt.format(n))
row("Ann. Sharpe", rep_gross.sharpe, rep_net.sharpe)
row("Sharpe CI low", rep_gross.sharpe_ci_low, rep_net.sharpe_ci_low)
row("Sharpe CI high", rep_gross.sharpe_ci_high, rep_net.sharpe_ci_high)
row("PSR(0)", rep_gross.psr, rep_net.psr)
row("MinTRL(0) bars", rep_gross.min_trl, rep_net.min_trl, "{:>14.1f}")
row("n_obs", rep_gross.n_obs, rep_net.n_obs, "{:>14d}")
row("Sortino", rep_gross.sortino, rep_net.sortino)
row("Calmar", rep_gross.calmar, rep_net.calmar)
row("Ann. return", rep_gross.ann_return, rep_net.ann_return)
row("Ann. vol", rep_gross.ann_vol, rep_net.ann_vol)
row("Max drawdown", rep_gross.max_drawdown, rep_net.max_drawdown)
row("Turnover (2-way)", rep_gross.turnover, rep_net.turnover)
row("Total costs ($)", rep_gross.total_costs, rep_net.total_costs, "{:>14,.0f}")


============================== GROSS ==============================
MetricsReport:
  Total return         +270.75%
  Annualized return    +26.92%
  Annualized vol       2.52%
  Hit rate             71.41%
  Total PnL ($)        +2,707,450.75
  Total costs ($)      0.00
  Gross PnL ($)        +2,707,450.75
  Turnover             86.05x
  Margin               1.4741%
  Sharpe (ann.)        9.156
  Sharpe SD (ann.)     0.438
  Sharpe 95% CI        [8.297, 10.015]
  Sortino              12.043
  Calmar               28.121
  Modified Sharpe      142.123
  PSR (vs 0)           1.000
  MinTRL (vs 0)        10 bars
  VaR 5%               -0.0016
  cVaR 5%              -0.0022
  Max drawdown         0.82%
  Time underwater      40.77%
  Longest underwater   23 bars
  N obs                1434
  Median cal-year ret  +25.44%
  Median arith ann     +22.81%

============================== NET ================================
MetricsReport:
  Total return         +228.96%
  Annualized return    +24

In [ ]:
# Cell 7 — Sabotage / control validation (spec test_universe.generalization)
# All GROSS (costs=None). These are diagnostic sign/magnitude checks that the
# +9.16 is a real edge and not PnL inflation / look-ahead. NOT deployable strategies.
import numpy as np, pandas as pd
from pathlib import Path
from backtest.data import DataPanel
from backtest.engine import Engine

DATA_DIR = Path(r"C:\Users\User\backtest_engine\backtest_engine2\data")


def ann_sharpe(res):
    r = res.returns.dropna()
    return r.mean() / r.std() * np.sqrt(252)


# control panel: same format, all drifts = 0 (pure random walk, seed 2)
px_ctrl = pd.read_parquet(DATA_DIR / "synth_edge_prices_control.parquet")
vol_ctrl = pd.read_parquet(DATA_DIR / "synth_edge_volume_control.parquet")
panel_ctrl = DataPanel(px_ctrl, volume=vol_ctrl, check_outliers=True)

# --- Check 1: control data, unchanged strategy -> expect Sharpe ~ 0 ---
res_ctrl = Engine(panel_ctrl, XSMomentum(60, 5), costs=None, liquidity=None).run(
    start=panel_ctrl.dates[WARMUP_BARS])
s_ctrl = ann_sharpe(res_ctrl)


# --- Check 2: sign-flipped strategy on RIGGED data -> expect ~ -9 ---
class XSMomentumFlipped(XSMomentum):
    def __repr__(self):
        return f"XSMomentumFlipped(lookback={self.lookback}, k={self.k})"

    def generate_weights(self, data, t):
        return -super().generate_weights(data, t)  # long losers, short winners


res_flip = Engine(panel, XSMomentumFlipped(60, 5), costs=None, liquidity=None).run(
    start=panel.dates[WARMUP_BARS])
s_flip = ann_sharpe(res_flip)


# --- Check 3: 1-bar LOOK-AHEAD on control data -> expect Sharpe to explode ---
# DELIBERATE LEAK: bypass DataView, read t+1 price as the signal. Diagnostic only.
class Lookahead(XSMomentum):
    def __init__(self, full_prices, lookback=60, k=5):
        super().__init__(lookback, k)
        self.full = full_prices

    def __repr__(self):
        return f"Lookahead(k={self.k})  # LEAK: peeks at t+1"

    def generate_weights(self, data, t):
        pos = self.full.index.get_loc(t)
        if pos + 1 >= len(self.full):
            return pd.Series(0.0, index=self.full.columns)
        fwd = self.full.iloc[pos + 1] / self.full.iloc[pos] - 1.0  # tomorrow's return (LEAK)
        ranked = fwd.rank()
        w = pd.Series(0.0, index=self.full.columns)
        w.loc[ranked.nlargest(self.k).index] = +0.5 / self.k
        w.loc[ranked.nsmallest(self.k).index] = -0.5 / self.k
        return w


res_look = Engine(panel_ctrl, Lookahead(px_ctrl, 60, 5), costs=None, liquidity=None).run(
    start=panel_ctrl.dates[WARMUP_BARS])
s_look = ann_sharpe(res_look)

print(f"{'check':<34}{'Sharpe':>9}   expected")
print("-" * 64)
print(f"{'1. control, normal strat':<34}{s_ctrl:>9.2f}   ~ 0    (|S| < 0.75)")
print(f"{'2. sign-flip on rigged':<34}{s_flip:>9.2f}   ~ -9   (mirror of +9.16)")
print(f"{'3. look-ahead on control':<34}{s_look:>9.2f}   >> 3   (leak detector)")
print()
print(f"reference: rigged normal strat gross Sharpe = {rep_gross.sharpe:+.2f}")


check                                Sharpe   expected
----------------------------------------------------------------
1. control, normal strat              -0.35   ~ 0    (|S| < 0.75)
2. sign-flip on rigged                -9.16   ~ -9   (mirror of +9.16)
3. look-ahead on control               6.54   >> 3   (leak detector)

reference: rigged normal strat gross Sharpe = +9.16
